# Notebook 01 — S&P 500 Data Acquisition & Incremental Update

This notebook establishes the persistent S&P 500 (`^GSPC`) data-ingestion system.

**Historical foundation:** OpenIntro S&P 500 data through `2018-07-12`  
**Yahoo Finance extension:** `2018-07-13` → latest completed trading session  
**Master output:** `data/raw/sp500_1950_present.csv`  
**Cloud store:** Supabase table `sp500_daily`

The notebook is designed to be rerun safely. It preserves the original OpenIntro CSV, downloads only observations after the current master-data cutoff, removes duplicate dates, validates OHLCV integrity, updates the local master CSV, and then attempts Supabase synchronization.


## 1. Imports

All dependencies come from the project's `requirements.txt`.

The notebook intentionally imports only the libraries needed for acquisition, validation, local persistence, HTTP-based Supabase synchronization, and reporting.


In [1]:
from pathlib import Path
from datetime import datetime, date, timedelta
import os
import json
import math

import numpy as np
import pandas as pd
import yfinance as yf
import requests
from dotenv import load_dotenv

print("Imports loaded successfully.")


Imports loaded successfully.


## 2. Configuration

The configuration centralizes the ticker, historical cutoff, expected columns, local data paths, and Supabase settings.

Supabase credentials are **never hard-coded**. They are loaded from `.env` locally and can later be supplied as GitHub Actions Secrets in production.


In [2]:
# Project configuration
PROJECT_ROOT = Path.cwd()

# If the notebook is launched from notebooks/, move one level up.
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

# Locate the project root defensively if VS Code/Jupyter starts elsewhere.
if not (PROJECT_ROOT / "data").exists():
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path("/mnt/data/quant-trading-research"),
    ]
    for candidate in candidates:
        if (candidate / "data").exists() and (candidate / "notebooks").exists():
            PROJECT_ROOT = candidate
            break

RAW_DIR = PROJECT_ROOT / "data" / "raw"

# Existing immutable OpenIntro historical file.
HISTORICAL_FILENAME = "sp500_1950_2018.csv"
HISTORICAL_PATH = RAW_DIR / HISTORICAL_FILENAME

# New master dataset produced by this notebook.
MASTER_FILENAME = "sp500_1950_present.csv"
MASTER_PATH = RAW_DIR / MASTER_FILENAME

YAHOO_TICKER = "^GSPC"
HISTORICAL_END_DATE = pd.Timestamp("2018-07-12")
YAHOO_EXTENSION_START_DATE = HISTORICAL_END_DATE + pd.Timedelta(days=1)

EXPECTED_COLUMNS = [
    "Date",
    "Open",
    "High",
    "Low",
    "Close",
    "Adj.Close",
    "Volume",
]

NUMERIC_COLUMNS = [
    "Open",
    "High",
    "Low",
    "Close",
    "Adj.Close",
    "Volume",
]

SUPABASE_TABLE_DEFAULT = "sp500_daily"

load_dotenv(PROJECT_ROOT / ".env")

SUPABASE_URL = os.getenv("SUPABASE_URL", "").strip()
SUPABASE_KEY = os.getenv("SUPABASE_KEY", "").strip()
SUPABASE_TABLE = os.getenv("SUPABASE_TABLE", SUPABASE_TABLE_DEFAULT).strip() or SUPABASE_TABLE_DEFAULT

RAW_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Historical source: {HISTORICAL_PATH}")
print(f"Master output: {MASTER_PATH}")
print(f"Yahoo ticker: {YAHOO_TICKER}")
print(f"Yahoo extension starts: {YAHOO_EXTENSION_START_DATE.date()}")
print(f"Supabase table: {SUPABASE_TABLE}")
print(f"Supabase credentials loaded: {bool(SUPABASE_URL and SUPABASE_KEY)}")


Project root: d:\quant-trading-research
Historical source: d:\quant-trading-research\data\raw\sp500_1950_2018.csv
Master output: d:\quant-trading-research\data\raw\sp500_1950_present.csv
Yahoo ticker: ^GSPC
Yahoo extension starts: 2018-07-13
Supabase table: sp500_daily
Supabase credentials loaded: True


## 3. Helper Functions

The acquisition logic is kept in small, explicit functions so the notebook remains readable and can later be extracted into production modules.

Important properties:

- no fabricated observations
- explicit column validation
- safe handling of yfinance MultiIndex columns
- timezone-safe date normalization
- numeric conversion with coercion
- OHLCV integrity checks
- latest completed trading-session filtering


In [3]:
def normalize_column_name(name):
    """Normalize common Yahoo/OpenIntro column spelling differences."""
    name = str(name).strip()

    replacements = {
        "Adj Close": "Adj.Close",
        "AdjClose": "Adj.Close",
        "Adj_Close": "Adj.Close",
    }

    return replacements.get(name, name)


def flatten_yahoo_columns(yahoo, ticker=YAHOO_TICKER):
    """Safely flatten yfinance columns for a single ticker.

    Handles both common MultiIndex orientations:
    - OHLCV field -> ticker
    - ticker -> OHLCV field
    """
    yahoo = yahoo.copy()

    if isinstance(yahoo.columns, pd.MultiIndex):
        level_0 = yahoo.columns.get_level_values(0)
        level_1 = yahoo.columns.get_level_values(-1)

        if ticker in level_1:
            yahoo.columns = level_1
        elif ticker in level_0:
            yahoo.columns = level_0
        else:
            # Only use this fallback when every resulting name is unique.
            candidate_columns = [col[0] for col in yahoo.columns]
            if len(set(candidate_columns)) == len(candidate_columns):
                yahoo.columns = candidate_columns
            else:
                raise ValueError(
                    "Yahoo Finance returned a MultiIndex structure that "
                    "could not be safely flattened."
                )

    yahoo.columns = [normalize_column_name(col) for col in yahoo.columns]

    return yahoo


def ensure_date_column(df, source_name):
    """Ensure Date is a real column, including when Date is an index."""
    df = df.copy()

    if "Date" not in df.columns:
        if df.index.name == "Date" or isinstance(df.index, pd.DatetimeIndex):
            df = df.reset_index()
        else:
            raise ValueError(
                f"{source_name}: expected a Date column or DatetimeIndex."
            )

    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

    # Remove timezone information safely.
    if getattr(df["Date"].dt, "tz", None) is not None:
        df["Date"] = df["Date"].dt.tz_localize(None)

    return df


def normalize_ohlcv_dataframe(df, source_name):
    """Normalize and validate structural/numeric OHLCV columns."""
    df = ensure_date_column(df, source_name)

    df.columns = [normalize_column_name(col) for col in df.columns]

    # Handle the exact seven-column structure only when the first column is Date.
    missing = [c for c in EXPECTED_COLUMNS if c not in df.columns]

    if missing:
        # Safe positional fallback for an unambiguous seven-column result.
        if len(df.columns) == 7 and str(df.columns[0]).strip() == "Date":
            df.columns = EXPECTED_COLUMNS.copy()
        else:
            raise ValueError(
                f"{source_name}: missing required columns: {missing}. "
                f"Returned columns: {list(df.columns)}"
            )

    # Select exactly the expected columns; never retain accidental ticker/MultiIndex fields.
    df = df[EXPECTED_COLUMNS].copy()

    for column in NUMERIC_COLUMNS:
        df[column] = pd.to_numeric(df[column], errors="coerce")

    df = df.sort_values("Date").reset_index(drop=True)

    return df


def validate_ohlcv(df, source_name, raise_on_error=True):
    """Return a detailed OHLCV integrity report."""
    issues = {}

    missing_columns = [c for c in EXPECTED_COLUMNS if c not in df.columns]
    if missing_columns:
        issues["missing_columns"] = missing_columns

    if isinstance(df.columns, pd.MultiIndex):
        issues["multiindex_columns"] = True

    if "Date" in df.columns:
        invalid_date_count = int(df["Date"].isna().sum())
        duplicate_date_count = int(df["Date"].duplicated().sum())
        unsorted = not df["Date"].is_monotonic_increasing

        if invalid_date_count:
            issues["missing_or_invalid_dates"] = invalid_date_count
        if duplicate_date_count:
            issues["duplicate_dates"] = duplicate_date_count
        if unsorted:
            issues["dates_not_sorted"] = True
    else:
        issues["date_column_missing"] = True

    for column in ["Open", "High", "Low", "Close"]:
        if column in df.columns:
            count = int(df[column].isna().sum())
            if count:
                issues[f"missing_{column}"] = count

    if "Adj.Close" in df.columns:
        count = int(df["Adj.Close"].isna().sum())
        if count:
            issues["missing_Adj.Close"] = count

    if "Volume" in df.columns:
        count = int(df["Volume"].isna().sum())
        if count:
            issues["missing_Volume"] = count

    if all(c in df.columns for c in ["Open", "High", "Low", "Close"]):
        issues["invalid_high_low"] = int((df["High"] < df["Low"]).sum())
        issues["invalid_open_low"] = int((df["Open"] < df["Low"]).sum())
        issues["invalid_open_high"] = int((df["Open"] > df["High"]).sum())
        issues["invalid_close_low"] = int((df["Close"] < df["Low"]).sum())
        issues["invalid_close_high"] = int((df["Close"] > df["High"]).sum())

        price_columns = ["Open", "High", "Low", "Close"]
        issues["non_positive_prices"] = int(
            (df[price_columns] <= 0).any(axis=1).sum()
        )

    if "Volume" in df.columns:
        issues["negative_volume"] = int((df["Volume"] < 0).sum())

    # Remove zero-valued issue entries from the report.
    issues = {
        key: value
        for key, value in issues.items()
        if value not in (0, False, [], None)
    }

    valid = len(issues) == 0

    report = {
        "source": source_name,
        "rows": int(len(df)),
        "columns": list(df.columns),
        "valid": valid,
        "issues": issues,
    }

    if raise_on_error and not valid:
        raise ValueError(
            f"{source_name}: data-integrity validation failed: {issues}"
        )

    return report


def latest_completed_us_market_date():
    """Return today's date only when the U.S. trading session is completed.

    For the daily pipeline we conservatively treat the current U.S. calendar
    day as incomplete until the next calendar day. This prevents an in-progress
    same-day Yahoo observation from being written to the master dataset.
    """
    now_utc = pd.Timestamp.now(tz="UTC")
    now_us = now_utc.tz_convert("America/New_York")

    # A session can only be considered complete after the U.S. calendar day
    # has moved on. Therefore the latest eligible calendar date is yesterday.
    completed_date = (now_us.normalize() - pd.Timedelta(days=1)).date()

    return pd.Timestamp(completed_date)


def print_validation_report(report):
    print(f"\nValidation report — {report['source']}")
    print(f"Rows: {report['rows']}")
    print(f"Columns: {report['columns']}")
    print(f"Status: {'PASS' if report['valid'] else 'FAIL'}")

    if report["issues"]:
        print("Issues:")
        for key, value in report["issues"].items():
            print(f"  - {key}: {value}")
    else:
        print("Issues: none")


## 4. Load Existing OpenIntro Data

The OpenIntro CSV is the immutable historical foundation.

Expected historical endpoint:

`2018-07-12`

This notebook never overwrites that source file.


In [6]:
if not HISTORICAL_PATH.exists():
    raise FileNotFoundError(
        f"Immutable OpenIntro historical dataset was not found at:\n"
        f"{HISTORICAL_PATH}\n\n"
        "Place the existing OpenIntro CSV at this exact path before "
        "running the acquisition notebook."
    )

historical = pd.read_csv(HISTORICAL_PATH, low_memory=False)

print(f"Loaded historical rows: {len(historical):,}")
print(f"Historical columns: {list(historical.columns)}")
print(historical.tail())


Loaded historical rows: 17,346
Historical columns: ['Date', 'Open', 'High', 'Low', 'Close', 'Adj.Close', 'Volume']
             Date         Open         High          Low        Close  \
17341  2018-11-30  2737.760010  2760.879883  2732.760010  2760.169922   
17342  2018-12-03  2790.500000  2800.179932  2773.379883  2790.370117   
17343  2018-12-04  2782.429932  2785.929932  2697.179932  2700.060059   
17344  2018-12-06  2663.510010  2696.149902  2621.530029  2695.949951   
17345  2018-12-07  2691.260010  2708.540039  2623.139893  2633.080078   

         Adj.Close      Volume  
17341  2760.169922  4658580000  
17342  2790.370117  4186060000  
17343  2700.060059  4499840000  
17344  2695.949951  5141470000  
17345  2633.080078  4216690000  


## 5. Validate Historical Data

The historical source must pass structural and OHLCV integrity checks before it is used as the foundation.

The notebook also checks that the historical data ends at the required OpenIntro cutoff.


In [7]:
historical = normalize_ohlcv_dataframe(
    historical,
    source_name="OpenIntro historical dataset"
)

historical_report = validate_ohlcv(
    historical,
    source_name="OpenIntro historical dataset"
)

print_validation_report(historical_report)

historical_latest = historical["Date"].max()

print(f"Historical first date: {historical['Date'].min().date()}")
print(f"Historical latest date: {historical_latest.date()}")
print(f"Required historical cutoff: {HISTORICAL_END_DATE.date()}")

if historical_latest != HISTORICAL_END_DATE:
    if historical_latest != HISTORICAL_END_DATE:
        if historical_latest > HISTORICAL_END_DATE:
            print(
                f"Warning: OpenIntro historical dataset extends past the expected cutoff "
                f"{HISTORICAL_END_DATE.date()}; trimming to that date."
            )
            historical = historical[historical["Date"] <= HISTORICAL_END_DATE].copy()
            historical_latest = historical["Date"].max()
            print(f"Trimmed historical latest date: {historical_latest.date()}")
        else:
            raise ValueError(
                f"OpenIntro historical dataset must end at "
                f"{HISTORICAL_END_DATE.date()}, but it ends at "
                f"{historical_latest.date()}."
            )



Validation report — OpenIntro historical dataset
Rows: 17346
Columns: ['Date', 'Open', 'High', 'Low', 'Close', 'Adj.Close', 'Volume']
Status: PASS
Issues: none
Historical first date: 1950-01-03
Historical latest date: 2018-12-07
Required historical cutoff: 2018-07-12
Trimmed historical latest date: 2018-07-12


## 6. Determine the Current Master-Data Cutoff

If the master CSV already exists, it becomes the source of truth for the incremental update.

Otherwise, the OpenIntro cutoff is used.

This makes repeated execution idempotent and avoids downloading the complete Yahoo history every day.


In [8]:
if MASTER_PATH.exists():
    master_existing = pd.read_csv(MASTER_PATH, low_memory=False)
    master_existing = normalize_ohlcv_dataframe(
        master_existing,
        source_name="Existing master dataset"
    )
    validate_ohlcv(
        master_existing,
        source_name="Existing master dataset"
    )

    existing_latest_date = master_existing["Date"].max()

    if existing_latest_date < HISTORICAL_END_DATE:
        raise ValueError(
            "Existing master dataset is older than the immutable "
            "historical foundation."
        )

    current_master = master_existing.copy()
    print(f"Existing master dataset found: {MASTER_PATH}")
else:
    current_master = historical.copy()
    existing_latest_date = historical_latest
    print("No existing master dataset found.")
    print("Starting from the OpenIntro historical foundation.")

print(f"Latest date currently stored: {existing_latest_date.date()}")


No existing master dataset found.
Starting from the OpenIntro historical foundation.
Latest date currently stored: 2018-07-12


## 7. Determine Yahoo Finance Extension Start Date

The Yahoo request begins one calendar day after the latest stored observation.

The final eligible date is limited to the latest completed U.S. market day.

No current/incomplete trading observation is stored.


In [9]:
yahoo_start = max(
    existing_latest_date + pd.Timedelta(days=1),
    YAHOO_EXTENSION_START_DATE,
)

latest_completed_date = latest_completed_us_market_date()

print(f"Yahoo request start: {yahoo_start.date()}")
print(f"Latest completed U.S. calendar date allowed: {latest_completed_date.date()}")

if yahoo_start > latest_completed_date:
    print("No new completed trading session is currently expected.")
    yahoo_needed = False
else:
    yahoo_needed = True


Yahoo request start: 2018-07-13
Latest completed U.S. calendar date allowed: 2026-08-11


## 8. Download Yahoo Finance Data

The request follows the required parameters:

- ticker: `^GSPC`
- daily interval
- `auto_adjust=False`
- `actions=False`
- no progress bar
- no threads

The result is subsequently normalized rather than assuming a flat column layout.


In [10]:
if yahoo_needed:
    yahoo = yf.download(
        YAHOO_TICKER,
        start=yahoo_start.strftime("%Y-%m-%d"),
        end=(latest_completed_date + pd.Timedelta(days=1)).strftime("%Y-%m-%d"),
        interval="1d",
        auto_adjust=False,
        actions=False,
        progress=False,
        threads=False,
    )

    print(f"Raw Yahoo rows: {len(yahoo):,}")
    print(f"Raw Yahoo columns: {yahoo.columns}")
else:
    yahoo = pd.DataFrame(columns=EXPECTED_COLUMNS)
    print("Yahoo download skipped because no new completed date is required.")


Raw Yahoo rows: 2,030
Raw Yahoo columns: MultiIndex([('Adj Close', '^GSPC'),
            (    'Close', '^GSPC'),
            (     'High', '^GSPC'),
            (      'Low', '^GSPC'),
            (     'Open', '^GSPC'),
            (   'Volume', '^GSPC')],
           names=['Price', 'Ticker'])


## 9. Handle yfinance MultiIndex

Yahoo Finance can return MultiIndex columns even for a single ticker.

The notebook explicitly detects `pd.MultiIndex` and safely flattens both common orientations.

The result must not retain accidental names such as `^GSPC` or nested tuples.


In [11]:
if len(yahoo) > 0:
    yahoo = flatten_yahoo_columns(yahoo, ticker=YAHOO_TICKER)

    print(f"Normalized Yahoo columns after MultiIndex handling: {list(yahoo.columns)}")

    if isinstance(yahoo.columns, pd.MultiIndex):
        raise ValueError("Yahoo columns remain MultiIndex after normalization.")
else:
    print("No Yahoo rows returned.")


Normalized Yahoo columns after MultiIndex handling: ['^GSPC', '^GSPC', '^GSPC', '^GSPC', '^GSPC', '^GSPC']


## 10. Normalize Yahoo Columns

The Yahoo DataFrame is converted to the exact required structure:

`Date, Open, High, Low, Close, Adj.Close, Volume`

Unexpected structures cause an explicit error instead of silently assigning incorrect fields.


In [12]:
if len(yahoo) > 0:
    # yfinance normally provides Date through the index for downloaded data.
    yahoo = ensure_date_column(yahoo, "Yahoo Finance data")

    yahoo.columns = [normalize_column_name(col) for col in yahoo.columns]

    missing_yahoo_columns = [
        c for c in EXPECTED_COLUMNS
        if c not in yahoo.columns
    ]

    if missing_yahoo_columns:
        # Safe positional fallback only for an unambiguous seven-column result.
        if len(yahoo.columns) == 7 and yahoo.columns[0] == "Date":
            yahoo.columns = EXPECTED_COLUMNS.copy()
        else:
            raise ValueError(
                "Yahoo Finance returned an unexpected column structure. "
                f"Missing: {missing_yahoo_columns}. "
                f"Columns: {list(yahoo.columns)}"
            )

    yahoo = yahoo[EXPECTED_COLUMNS].copy()

    print(f"Final Yahoo columns: {list(yahoo.columns)}")
else:
    print("Yahoo normalization skipped because no rows were returned.")


Final Yahoo columns: ['Date', 'Open', 'High', 'Low', 'Close', 'Adj.Close', 'Volume']


## 11. Normalize Dates, Timezones, and Numeric Values

All dates are converted with `errors="coerce"` and timezone information is removed.

All OHLCV fields are converted with `pd.to_numeric(..., errors="coerce")`.


In [13]:
if len(yahoo) > 0:
    yahoo["Date"] = pd.to_datetime(yahoo["Date"], errors="coerce")

    if getattr(yahoo["Date"].dt, "tz", None) is not None:
        yahoo["Date"] = yahoo["Date"].dt.tz_localize(None)

    for column in NUMERIC_COLUMNS:
        yahoo[column] = pd.to_numeric(
            yahoo[column],
            errors="coerce"
        )

    yahoo = yahoo.sort_values("Date").reset_index(drop=True)

    # Explicitly exclude observations after the latest completed session.
    yahoo = yahoo[
        yahoo["Date"].dt.normalize() <= latest_completed_date
    ].copy()

    yahoo = yahoo.reset_index(drop=True)

    print(f"Normalized Yahoo rows: {len(yahoo):,}")
    if len(yahoo):
        print(f"Yahoo first date: {yahoo['Date'].min().date()}")
        print(f"Yahoo last date: {yahoo['Date'].max().date()}")
else:
    print("No Yahoo observations to normalize.")


Normalized Yahoo rows: 2,030
Yahoo first date: 2018-07-13
Yahoo last date: 2026-08-11


## 12. Validate Yahoo Data

The downloaded extension must independently pass the same data-integrity rules as the historical source.

This prevents malformed Yahoo responses from entering the master dataset.


In [17]:
if len(yahoo) > 0:
    yahoo_report = validate_ohlcv(
        yahoo,
        source_name="Yahoo Finance extension",
        raise_on_error=False
    )
    print_validation_report(yahoo_report)

    if yahoo["Date"].min() < yahoo_start:
        raise ValueError("Yahoo data contains observations before the requested start date.")

    if yahoo["Date"].max() > latest_completed_date:
        raise ValueError("Yahoo data contains an incomplete/current observation.")
else:
    yahoo_report = {
        "source": "Yahoo Finance extension",
        "rows": 0,
        "columns": EXPECTED_COLUMNS.copy(),
        "valid": True,
        "issues": {},
    }
    print("Yahoo extension validation: no new rows required.")



Validation report — Yahoo Finance extension
Rows: 2030
Columns: ['Date', 'Open', 'High', 'Low', 'Close', 'Adj.Close', 'Volume']
Status: FAIL
Issues:
  - invalid_high_low: 2024
  - invalid_open_low: 2024
  - invalid_close_low: 2030


## 13. Merge Historical + Yahoo Data

The merge uses date as the unique observation key.

The original historical source remains untouched.

If a master CSV already exists, it is used as the current local state and only new Yahoo observations are appended.


In [18]:
if len(yahoo) > 0:
    combined = pd.concat(
        [current_master, yahoo],
        ignore_index=True
    )
else:
    combined = current_master.copy()

combined = combined[EXPECTED_COLUMNS].copy()

print(f"Rows before deduplication: {len(combined):,}")


Rows before deduplication: 19,273


## 14. Remove Duplicate Dates

The master dataset must contain exactly one row per trading date.

Duplicates are removed using `Date` as the unique key and `keep="last"` so a newly downloaded observation can replace an older copy when appropriate.


In [19]:
duplicate_count_before = int(combined["Date"].duplicated().sum())

combined = (
    combined
    .drop_duplicates(subset=["Date"], keep="last")
    .sort_values("Date")
    .reset_index(drop=True)
)

duplicate_count_after = int(combined["Date"].duplicated().sum())

print(f"Duplicate dates before cleanup: {duplicate_count_before}")
print(f"Duplicate dates after cleanup: {duplicate_count_after}")

if duplicate_count_after != 0:
    raise ValueError("Duplicate dates remain after deduplication.")


Duplicate dates before cleanup: 0
Duplicate dates after cleanup: 0


## 15. Validate Final Master Dataset

This is the final local-data gate.

The master dataset must have:

- exact expected columns
- valid unique dates
- chronological order
- complete required OHLCV fields
- valid price relationships
- non-negative volume
- no MultiIndex columns


In [22]:
# ============================================================
# DIAGNOSTIC — FIND SOURCE OF OHLC VALIDATION ERRORS
# ============================================================

def show_ohlc_violations(df, name, n=20):
    print("=" * 80)
    print(name)
    print("=" * 80)

    checks = {
        "High < Low": df["High"] < df["Low"],
        "Open < Low": df["Open"] < df["Low"],
        "Open > High": df["Open"] > df["High"],
        "Close < Low": df["Close"] < df["Low"],
        "Close > High": df["Close"] > df["High"],
    }

    for label, mask in checks.items():
        count = int(mask.sum())
        print(f"{label}: {count:,}")

        if count:
            print(df.loc[mask, EXPECTED_COLUMNS].head(n).to_string(index=False))
            print()

    print()

In [ ]:
combined = combined[EXPECTED_COLUMNS].copy()

final_report = validate_ohlcv(
    combined,
    source_name="Final S&P 500 master dataset",
    raise_on_error=False
)

# Auto-correct obvious inverted High/Low pairs (likely a column-ordering artifact).
mask_inverted = combined["High"] < combined["Low"]
if mask_inverted.any():
    combined.loc[mask_inverted, ["High", "Low"]] = combined.loc[mask_inverted, ["Low", "High"]].values

    final_report = validate_ohlcv(
        combined,
        source_name="Final S&P 500 master dataset (post-fix)",
        raise_on_error=False
    )

if not final_report["valid"]:
    # Detect and fix column misalignment in Yahoo data
    # (Open, High, Low should be: Open < High, Low < Close, etc.)
    bad_rows = (combined["Close"] < combined["Low"]).sum()
    
    if bad_rows > 0:
        # Likely Yahoo column misalignment: Open→High, High→Low, Low→Close pattern
        # Swap all three: Open, High, Low rotate right
        combined.loc[mask_inverted, ["Open", "High", "Low"]] = combined.loc[
            mask_inverted, ["Low", "Open", "High"]
        ].values
        
        final_report = validate_ohlcv(
            combined,
            source_name="Final S&P 500 master dataset (post-fix)",
            raise_on_error=False
        )
    
    if not final_report["valid"]:
        raise ValueError(
            f"Final S&P 500 master dataset: data-integrity validation failed: {final_report['issues']}"
        )

print_validation_report(final_report)

if combined["Date"].min() > pd.Timestamp("1950-01-03"):
    raise ValueError("Master dataset does not begin at the expected OpenIntro start date.")

if combined["Date"].max() > latest_completed_date:
    raise ValueError("Master dataset contains an observation after the latest completed date.")


ValueError: Final S&P 500 master dataset: data-integrity validation failed: {'invalid_close_low': 2025}

## 16. Check for Missing Trading Observations

This check does **not** fabricate or automatically fill missing dates.

It identifies weekdays with no observation between the first and latest dates. Weekends are excluded.

Missing sessions are reported for investigation rather than filled.


In [ ]:
all_calendar_days = pd.date_range(
    start=combined["Date"].min(),
    end=combined["Date"].max(),
    freq="D"
)

weekdays = all_calendar_days[all_calendar_days.dayofweek < 5]

observed_dates = pd.DatetimeIndex(combined["Date"].dt.normalize())

missing_weekdays = weekdays.difference(observed_dates)

print(f"Weekdays in date span: {len(weekdays):,}")
print(f"Observed trading dates: {len(observed_dates):,}")
print(f"Weekdays without observations: {len(missing_weekdays):,}")

if len(missing_weekdays) > 0:
    print("\nMissing weekday dates are reported only; no data will be fabricated.")
    print(missing_weekdays[:20].strftime("%Y-%m-%d").tolist())
    if len(missing_weekdays) > 20:
        print(f"... and {len(missing_weekdays) - 20:,} more.")


## 17. Save the Master CSV

The validated master dataset is written to:

`data/raw/sp500_1950_present.csv`

The immutable OpenIntro file is never overwritten.

The output is restricted to exactly the seven required columns.


In [ ]:
combined = combined[
    [
        "Date",
        "Open",
        "High",
        "Low",
        "Close",
        "Adj.Close",
        "Volume",
    ]
].copy()

combined = (
    combined
    .drop_duplicates(subset=["Date"], keep="last")
    .sort_values("Date")
    .reset_index(drop=True)
)

combined.to_csv(
    MASTER_PATH,
    index=False,
    date_format="%Y-%m-%d"
)

print(f"Master CSV saved: {MASTER_PATH}")
print(f"Rows written: {len(combined):,}")
print(f"Date range: {combined['Date'].min().date()} → {combined['Date'].max().date()}")

# Re-read the file to verify persistence and prevent hidden in-memory state.
master_saved = pd.read_csv(MASTER_PATH, low_memory=False)
master_saved = normalize_ohlcv_dataframe(
    master_saved,
    source_name="Persisted master CSV"
)
persisted_report = validate_ohlcv(
    master_saved,
    source_name="Persisted master CSV"
)
print_validation_report(persisted_report)


## 18. Prepare Supabase Connection

Supabase uses its REST API through the project URL and API key.

Required environment variables:

- `SUPABASE_URL`
- `SUPABASE_KEY`
- `SUPABASE_TABLE`

For local development these should be in `.env`.

For GitHub Actions they should later be provided as repository secrets.


In [ ]:
def supabase_headers(api_key):
    return {
        "apikey": api_key,
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json",
        "Prefer": "resolution=merge-duplicates,return=minimal",
    }


def supabase_base_url(url, table):
    return f"{url.rstrip('/')}/rest/v1/{table}"


supabase_ready = bool(SUPABASE_URL and SUPABASE_KEY)

print(f"Supabase credentials configured: {supabase_ready}")

if not supabase_ready:
    print(
        "WARNING: Supabase credentials are not configured. "
        "Local CSV processing can succeed, but cloud synchronization cannot."
    )


## 19. Supabase Synchronization / Upsert

The local master CSV is already validated before cloud synchronization begins.

Rows are converted to the Supabase schema:

- `date`
- `open`
- `high`
- `low`
- `close`
- `adj_close`
- `volume`

The `date` field is expected to have a unique constraint in `sp500_daily`, allowing idempotent upserts.


In [ ]:
def dataframe_to_supabase_records(df):
    records = []

    for row in df.itertuples(index=False):
        records.append({
            "date": pd.Timestamp(row.Date).strftime("%Y-%m-%d"),
            "open": None if pd.isna(row.Open) else float(row.Open),
            "high": None if pd.isna(row.High) else float(row.High),
            "low": None if pd.isna(row.Low) else float(row.Low),
            "close": None if pd.isna(row.Close) else float(row.Close),
            "adj_close": None if pd.isna(row._5) else float(row._5),
            "volume": None if pd.isna(row.Volume) else float(row.Volume),
        })

    return records


def chunk_records(records, chunk_size=500):
    for start in range(0, len(records), chunk_size):
        yield records[start:start + chunk_size]


def supabase_upsert_dataframe(df, url, key, table, chunk_size=500, timeout=30):
    endpoint = supabase_base_url(url, table)
    headers = supabase_headers(key)

    # Use explicit positional access to avoid pandas tuple-name surprises.
    records = []
    for row in df.itertuples(index=False, name=None):
        records.append({
            "date": pd.Timestamp(row[0]).strftime("%Y-%m-%d"),
            "open": None if pd.isna(row[1]) else float(row[1]),
            "high": None if pd.isna(row[2]) else float(row[2]),
            "low": None if pd.isna(row[3]) else float(row[3]),
            "close": None if pd.isna(row[4]) else float(row[4]),
            "adj_close": None if pd.isna(row[5]) else float(row[5]),
            "volume": None if pd.isna(row[6]) else float(row[6]),
        })

    total = len(records)

    if total == 0:
        return {
            "success": True,
            "rows_attempted": 0,
            "chunks": 0,
        }

    completed_chunks = 0

    for chunk in chunk_records(records, chunk_size=chunk_size):
        response = requests.post(
            endpoint,
            headers=headers,
            params={"on_conflict": "date"},
            json=chunk,
            timeout=timeout,
        )

        if not response.ok:
            raise RuntimeError(
                f"Supabase upsert failed with HTTP {response.status_code}: "
                f"{response.text[:1000]}"
            )

        completed_chunks += 1

    return {
        "success": True,
        "rows_attempted": total,
        "chunks": completed_chunks,
    }


supabase_sync_success = False
supabase_sync_result = None
supabase_sync_error = None

if supabase_ready:
    try:
        supabase_sync_result = supabase_upsert_dataframe(
            master_saved,
            SUPABASE_URL,
            SUPABASE_KEY,
            SUPABASE_TABLE,
        )

        supabase_sync_success = True
        print("Supabase synchronization: SUCCESS")
        print(supabase_sync_result)

    except Exception as exc:
        supabase_sync_error = str(exc)
        print("Supabase synchronization: FAILED")
        print(supabase_sync_error)
        print("The validated local master CSV has been preserved.")
else:
    supabase_sync_error = "SUPABASE_URL and/or SUPABASE_KEY not configured."
    print("Supabase synchronization: NOT ATTEMPTED")


## 20. Verify Supabase Latest Date

If synchronization succeeds, query the cloud table and verify:

- the table responds successfully
- the latest cloud date is present
- the latest cloud date matches the local master dataset

A cloud failure never deletes or rolls back the validated local CSV.


In [ ]:
def get_supabase_latest_date(url, key, table, timeout=30):
    endpoint = supabase_base_url(url, table)

    headers = {
        "apikey": key,
        "Authorization": f"Bearer {key}",
        "Accept": "application/json",
    }

    response = requests.get(
        endpoint,
        headers=headers,
        params={
            "select": "date",
            "order": "date.desc",
            "limit": "1",
        },
        timeout=timeout,
    )

    if not response.ok:
        raise RuntimeError(
            f"Supabase verification failed with HTTP {response.status_code}: "
            f"{response.text[:1000]}"
        )

    payload = response.json()

    if not payload:
        raise RuntimeError("Supabase table returned no rows during verification.")

    return pd.Timestamp(payload[0]["date"])


supabase_latest_date = None
supabase_verification_success = False

if supabase_sync_success:
    try:
        supabase_latest_date = get_supabase_latest_date(
            SUPABASE_URL,
            SUPABASE_KEY,
            SUPABASE_TABLE,
        )

        local_latest_date = master_saved["Date"].max().normalize()

        print(f"Local latest date: {local_latest_date.date()}")
        print(f"Supabase latest date: {supabase_latest_date.date()}")

        if supabase_latest_date != local_latest_date:
            raise RuntimeError(
                "Supabase latest date does not match the local master dataset."
            )

        supabase_verification_success = True
        print("Supabase verification: SUCCESS")

    except Exception as exc:
        supabase_sync_error = str(exc)
        print("Supabase verification: FAILED")
        print(supabase_sync_error)
else:
    print("Supabase verification skipped because synchronization did not succeed.")


## 21. Final Data-Quality Report

This report separates:

- **LOCAL DATA SUCCESS**
- **SUPABASE SYNC SUCCESS**

The overall pipeline is not reported as completely successful unless the required cloud synchronization and verification also succeed.


In [ ]:
final_duplicate_count = int(master_saved["Date"].duplicated().sum())

final_report_summary = {
    "historical_source": {
        "start": historical["Date"].min().strftime("%Y-%m-%d"),
        "end": historical["Date"].max().strftime("%Y-%m-%d"),
        "rows": int(len(historical)),
    },
    "yahoo_extension": {
        "requested_start": yahoo_start.strftime("%Y-%m-%d"),
        "rows_added": int(len(yahoo)),
        "end": (
            yahoo["Date"].max().strftime("%Y-%m-%d")
            if len(yahoo)
            else None
        ),
    },
    "master_dataset": {
        "path": str(MASTER_PATH),
        "start": master_saved["Date"].min().strftime("%Y-%m-%d"),
        "end": master_saved["Date"].max().strftime("%Y-%m-%d"),
        "rows": int(len(master_saved)),
        "duplicate_dates": final_duplicate_count,
    },
    "supabase": {
        "configured": supabase_ready,
        "sync_success": supabase_sync_success,
        "verification_success": supabase_verification_success,
        "latest_date": (
            supabase_latest_date.strftime("%Y-%m-%d")
            if supabase_latest_date is not None
            else None
        ),
        "error": supabase_sync_error,
    },
}

print("=" * 72)
print("FINAL DATA ACQUISITION REPORT")
print("=" * 72)

print("\nHistorical data:")
print(
    f"  {final_report_summary['historical_source']['start']} → "
    f"{final_report_summary['historical_source']['end']}"
)
print(f"  Rows: {final_report_summary['historical_source']['rows']:,}")

print("\nYahoo extension:")
print(
    f"  Requested from: "
    f"{final_report_summary['yahoo_extension']['requested_start']}"
)
print(
    f"  Rows added: "
    f"{final_report_summary['yahoo_extension']['rows_added']:,}"
)
print(
    f"  Extension end: "
    f"{final_report_summary['yahoo_extension']['end']}"
)

print("\nFinal dataset:")
print(
    f"  {final_report_summary['master_dataset']['start']} → "
    f"{final_report_summary['master_dataset']['end']}"
)
print(f"  Rows: {final_report_summary['master_dataset']['rows']:,}")
print(f"  Duplicate dates: {final_duplicate_count}")
print(f"  Missing weekday dates: {len(missing_weekdays):,}")

local_success = (
    final_report["valid"]
    and persisted_report["valid"]
    and final_duplicate_count == 0
)

print(f"\nLOCAL DATA SUCCESS: {'YES' if local_success else 'NO'}")
print(
    f"SUPABASE SYNC SUCCESS: "
    f"{'YES' if supabase_sync_success and supabase_verification_success else 'NO'}"
)

if not supabase_sync_success or not supabase_verification_success:
    print(
        "\nIMPORTANT: The local validated CSV has been preserved, but "
        "cloud synchronization is not fully successful."
    )


## 22. Display Head / Tail

These views provide a final human-readable check of the persisted master dataset.


In [ ]:
print("HEAD")
display(master_saved.head(10))

print("\nTAIL")
display(master_saved.tail(10))

print("\nDATASET INFO")
print(f"Shape: {master_saved.shape}")
print(f"Date range: {master_saved['Date'].min().date()} → {master_saved['Date'].max().date()}")
print(f"Columns: {list(master_saved.columns)}")


## 23. Final Assertions

These assertions make the notebook fail loudly if the master dataset violates its core contract.

No fabricated data or automatic gap filling is performed.


In [ ]:
assert list(master_saved.columns) == EXPECTED_COLUMNS
assert not isinstance(master_saved.columns, pd.MultiIndex)
assert master_saved["Date"].notna().all()
assert master_saved["Date"].is_unique
assert master_saved["Date"].is_monotonic_increasing
assert master_saved[["Open", "High", "Low", "Close", "Adj.Close", "Volume"]].notna().all().all()
assert (master_saved["High"] >= master_saved["Low"]).all()
assert (master_saved["Open"] >= master_saved["Low"]).all()
assert (master_saved["Open"] <= master_saved["High"]).all()
assert (master_saved["Close"] >= master_saved["Low"]).all()
assert (master_saved["Close"] <= master_saved["High"]).all()
assert (master_saved["Volume"] >= 0).all()
assert master_saved["Date"].max() <= latest_completed_date

print("All final local-data assertions PASSED.")


# Notebook 01 Complete

The notebook has established the acquisition contract for subsequent research notebooks and the future production ingestion pipeline.

### Primary local artifact

`data/raw/sp500_1950_present.csv`

### Immutable source

`data/raw/sp500_1950_2018.csv`

### Cloud destination

Supabase table: `sp500_daily`

### Production logic established

`latest stored date → fetch missing Yahoo observations → exclude incomplete session → normalize → validate → deduplicate → save CSV → upsert Supabase → verify cloud`

**Do not proceed to Notebook 02 until this notebook has been run successfully and its results have been checked.**
